Example Simple Chat bot Gateway V4

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
import os
import sqlite3
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import base64
from io import BytesIO
import dropbox
import requests
import json
from typing import Dict, List, Any, Optional
import pandas as pd
import requests
import json
from typing import Dict, List, Any, Optional
import pandas as pd
import ssl
import urllib3
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from typing import Dict, List, Any, Optional

In [2]:
load_dotenv(override=True)
openai = OpenAI()


In [3]:
def create_who_ssl_context():
    """
    Create a custom SSL context for WHO API connections that handles missing intermediate certificates
    
    This is a workaround for the WHO server not sending the complete certificate chain.
    The server certificate is valid but missing intermediate certificates.
    """
    # Create SSL context with custom settings for WHO domain
    ssl_context = ssl.create_default_context()
    
    # Disable hostname checking and certificate verification for WHO domains
    # This is necessary because the server doesn't send complete certificate chain
    ssl_context.check_hostname = False
    ssl_context.verify_mode = ssl.CERT_NONE
    
    # Suppress SSL warnings for this specific case
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    
    return ssl_context

def fetch_who_measure_data(indicator_code: str, country_groups: List[str] = None, countries: List[str] = None) -> Dict[str, Any]:
    """
    Fetch data from WHO European Health Information Gateway API for a specific measure
    
    Args:
        indicator_code: The measure code (e.g., 'hfa_43')
        country_groups: List of country group codes (e.g., ['WHO_EURO', 'EU_MEMBERS'])
        countries: List of country ISO codes (e.g., ['ITA', 'MDA'])
    
    Returns:
        Parsed JSON data from the 'data' field, or empty dict if error
    """
    base_url = "https://dw.euro.who.int/api/v3/measures/"
    url = f"{base_url}{indicator_code}"
    
    # Build filter parameters
    filter_parts = []
    
    if country_groups:
        country_groups_str = ",".join(country_groups)
        filter_parts.append(f"COUNTRY_GRP:{country_groups_str}")
    
    if countries:
        countries_str = ",".join(countries)
        filter_parts.append(f"COUNTRY:{countries_str}")
    
    # Manually construct URL to avoid automatic encoding of colons
    if filter_parts:
        filter_string = ";".join(filter_parts)
        url = f"{url}?filter={filter_string}"
    
    # Create browser-like headers to avoid 403 errors
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': 'https://gateway.euro.who.int/',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Cache-Control': 'max-age=0',
        'Sec-Fetch-Dest': 'document',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-Site': 'none'
    }
    
    try:
        # Create a session to maintain cookies from the beginning
        session = requests.Session()
        
        # Create custom SSL context for WHO connections
        ssl_context = create_who_ssl_context()
        
        # Configure session to use custom SSL context
        adapter = HTTPAdapter()
        session.mount('https://', adapter)
        
        # First access the main WHO Data Warehouse site to establish session
        main_url = "https://dw.euro.who.int"
        print(f"Establishing session with {main_url}...")
        session.get(main_url, headers=headers, timeout=30, verify=False)
        
        # Then make the API request using the established session
        print(f"Making API request to {url}...")
        response = session.get(url, headers=headers, timeout=30, verify=False)
        response.raise_for_status()
        json_data = response.json()
        
        # Return the 'data' field from the JSON response
        return json_data.get("data", {})
        
    except requests.exceptions.RequestException as e:
        print(f"API request failed: {e}")
        return {}
    except json.JSONDecodeError as e:
        print(f"JSON parsing failed: {e}")
        return {}
    except Exception as e:
        print(f"Unexpected error: {e}")
        return {}

# Example usage:
# data = fetch_who_measure_data("hfa_43", ["WHO_EURO", "EU_MEMBERS"], ["ITA", "MDA"])

In [4]:
def parse_who_data_structure(data: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Parse and format WHO API data structure for easier analysis
    
    Args:
        data: Raw data from WHO API (json_data["data"])
    
    Returns:
        List of formatted data records
    """
    if not data:
        return []
    
    # Handle different possible data structures
    if isinstance(data, list):
        return data
    elif isinstance(data, dict):
        # If data is a dict, look for common keys that contain the actual data
        for key in ["values", "observations", "data", "records"]:
            if key in data and isinstance(data[key], list):
                return data[key]
        # If no list found, return the dict as a single item
        return [data]
    
    return []

def parse_who_json_to_dataframe(json_data: Dict[str, Any]) -> 'pd.DataFrame':
    """
    Parse WHO API JSON response and extract data into a pandas DataFrame
    
    Args:
        json_data: Raw JSON response from WHO API (can be dict or list)
    
    Returns:
        pandas DataFrame with extracted data
    """
    
    if not json_data:
        return pd.DataFrame()
    
    # Handle both dict and list inputs
    if isinstance(json_data, list):
        # If json_data is already a list, use it directly
        data_records = json_data
    elif isinstance(json_data, dict):
        # If json_data is a dict, extract the data array
        data_records = json_data.get("data", [])
    else:
        return pd.DataFrame()
    
    if not data_records:
        return pd.DataFrame()
    
    # List to store flattened records
    flattened_records = []
    
    for record in data_records:
        if not isinstance(record, dict):
            continue
            
        # Extract dimensions (metadata about the record)
        dimensions = record.get("dimensions", {})
        
        # Extract value information
        value_info = record.get("value", {})
        
        # Create a flattened record
        flattened_record = {
            # Extract dimension fields
            "COUNTRY": dimensions.get("COUNTRY", ""),
            "COUNTRY_GRP": dimensions.get("COUNTRY_GRP", ""),
            "SEX": dimensions.get("SEX", ""),
            "YEAR": dimensions.get("YEAR", ""),
            
            # Extract value fields
            "VALUE_DISPLAY": value_info.get("display", ""),
            "VALUE_NUMERIC": value_info.get("numeric", None),
            
            # Keep original record for reference
            "RAW_RECORD": record
        }
        
        # Add any other dimension fields that might exist
        for key, value in dimensions.items():
            if key not in ["COUNTRY", "COUNTRY_GRP", "SEX", "YEAR"]:
                flattened_record[f"DIM_{key}"] = value
        
        flattened_records.append(flattened_record)
    
    # Create DataFrame
    df = pd.DataFrame(flattened_records)
    
    # Convert numeric columns to appropriate types
    if not df.empty:
        # Convert YEAR to integer if possible
        df["YEAR"] = pd.to_numeric(df["YEAR"], errors='coerce').astype('Int64')
        
        # Convert VALUE_NUMERIC to float
        df["VALUE_NUMERIC"] = pd.to_numeric(df["VALUE_NUMERIC"], errors='coerce')
        
        # Sort by COUNTRY and YEAR for better organization
        df = df.sort_values(["COUNTRY", "YEAR"], na_position='last')
    
    return df

In [5]:
# Get raw JSON data
raw_data = fetch_who_measure_data("hfa_43", ["WHO_EURO"], ["ITA", "MDA", "DNK"])

# Parse to DataFrame
df = parse_who_json_to_dataframe(raw_data)

# Now you can work with the data
print(df.head())
print(df.describe())
print(df.groupby('COUNTRY')['VALUE_NUMERIC'].mean())


Establishing session with https://dw.euro.who.int...
Making API request to https://dw.euro.who.int/api/v3/measures/hfa_43?filter=COUNTRY_GRP:WHO_EURO;COUNTRY:ITA,MDA,DNK...
    COUNTRY COUNTRY_GRP  SEX  YEAR VALUE_DISPLAY  VALUE_NUMERIC  \
142            WHO_EURO  ALL  1970          71.4          71.39   
143            WHO_EURO  ALL  1971          71.5          71.55   
144            WHO_EURO  ALL  1972          71.6          71.64   
145            WHO_EURO  ALL  1973          71.7          71.66   
146            WHO_EURO  ALL  1974          71.9          71.86   

                                            RAW_RECORD  
142  {'fact_id': '52999013', 'attributes': {'MEASUR...  
143  {'fact_id': '52999014', 'attributes': {'MEASUR...  
144  {'fact_id': '52999015', 'attributes': {'MEASUR...  
145  {'fact_id': '52999016', 'attributes': {'MEASUR...  
146  {'fact_id': '52999017', 'attributes': {'MEASUR...  
              YEAR  VALUE_NUMERIC
count        192.0     192.000000
mean   1995.61

In [6]:
#code to functions to search ind.db to register as a tool for chat agent
# Method to search by indicator name
def search_ind_name(ind_name: str) -> list | None:
    """ Get the list of URLS for given question about indicators """
    conn = sqlite3.connect("intro/ind.db")
    c = conn.cursor()
    # Use wildcards for partial matching and case-insensitive search
    search_pattern = f"%{ind_name.lower()}%"
    c.execute("SELECT name, url, indicator_code FROM indicators WHERE LOWER(name) LIKE ?", (search_pattern,))
    result = c.fetchall()
    conn.close()
    return result if result else None

# Enhanced search function with more options
def search_indicators_advanced(search_term: str, exact_match: bool = False) -> list | None:
    """ Advanced search with options for exact or partial matching """
    conn = sqlite3.connect("intro/ind.db")
    c = conn.cursor()
    
    if exact_match:
        # Exact match (case-insensitive)
        c.execute("SELECT name, url, indicator_code FROM indicators WHERE LOWER(name) = LOWER(?)", (search_term,))
    else:
        # Partial match with wildcards
        search_pattern = f"%{search_term.lower()}%"
        c.execute("SELECT name, url, indicator_code FROM indicators WHERE LOWER(name) LIKE ?", (search_pattern,))
    
    result = c.fetchall()
    print(str(result.count))
    conn.close()
    return result if result else None


In [7]:
def upload_image_to_dropbox(image_data, filename):
    """Upload image to Dropbox and return shareable link"""
    try:
        # Try OAuth with refresh token first (recommended)
        app_key = os.getenv('DROPBOX_APP_KEY')
        app_secret = os.getenv('DROPBOX_APP_SECRET')
        refresh_token = os.getenv('DROPBOX_REFRESH_TOKEN')
        
        if app_key and app_secret and refresh_token:
            # Use OAuth with refresh token (automatically refreshes when expired)
            dbx = dropbox.Dropbox(
                app_key=app_key,
                app_secret=app_secret,
                oauth2_refresh_token=refresh_token
            )
        else:
            # Fallback to simple access token (will expire after 4 hours)
            access_token = os.getenv('DROPBOX_ACCESS_TOKEN')
            if not access_token:
                return "Error: DROPBOX_ACCESS_TOKEN or OAuth credentials not found in environment variables"
            dbx = dropbox.Dropbox(access_token)
        
        # Upload file to Dropbox app folder
        # Automatically uploads to /Apps/YourAppName/filename
        dbx.files_upload(image_data, f'/{filename}')
        
        # Create shareable link (correct method name)
        shared_link = dbx.sharing_create_shared_link_with_settings(f'/{filename}')
        
        # Return the URL
        return shared_link.url
        
    except dropbox.exceptions.AuthError as e:
        return f"Dropbox authentication error: {str(e)}"
    except dropbox.exceptions.ApiError as e:
        return f"Dropbox API error: {str(e)}"
    except Exception as e:
        return f"Unexpected error: {str(e)}"

In [8]:
def convert_dropbox_url_to_direct(dropbox_url):
    """Convert Dropbox shareable URL to direct image URL for display"""
    if not dropbox_url or "Error:" in dropbox_url:
        return dropbox_url
    
    # Handle new Dropbox URL format: https://www.dropbox.com/scl/fi/abc123/filename.png?rlkey=xyz&dl=0
    if "dropbox.com/scl/fi/" in dropbox_url:
        # Extract the file ID from the URL
        import re
        match = re.search(r'/scl/fi/([^/]+)/', dropbox_url)
        if match:
            file_id = match.group(1)
            # Create direct URL format
            direct_url = f"https://dl.dropboxusercontent.com/scl/fi/{file_id}/"
            return direct_url
    
    # Handle old format: https://www.dropbox.com/s/abc123/filename.png?dl=0
    elif "dropbox.com/s/" in dropbox_url:
        direct_url = dropbox_url.replace("www.dropbox.com/s/", "dl.dropboxusercontent.com/s/")
        direct_url = direct_url.split("?")[0]  # Remove query parameters
        return direct_url
    
    return dropbox_url

In [9]:
def get_country_full_name(iso3_code: str) -> str:
    """
    Get full country name from ISO3 code using the euro_countries data
    """
    try:
        # Load the countries data
        with open("intro/euro_countries.json", "r", encoding="utf-8") as f:
            countries_data = json.loads(f.read())
        
        # Find the country by ISO3 code
        for country in countries_data:
            if country.get("iso3") == iso3_code or country.get("code") == iso3_code:
                return country.get("full_name", iso3_code)
        
        # If not found, return the ISO3 code
        return iso3_code
    except Exception:
        # If any error, return the ISO3 code
        return iso3_code

def create_plot(indicator_code: str, indicator_name: str, countries: List[str] = None, country_groups: List[str] = None) -> str:
    """
    Create a plot from WHO API data and upload to Dropbox
    
    Args:
        indicator_code: The measure code (e.g., 'hfa_43')
        indicator_name: Display name for the indicator (e.g., 'Life Expectancy at Birth')
        countries: List of country ISO3 codes (e.g., ['ITA', 'MDA', 'DNK'])
        country_groups: List of country group codes (e.g., ['WHO_EURO', 'EU_MEMBERS'])
    
    Returns:
        Direct URL to the uploaded plot image
    """
    try:
        # Get raw JSON data
        print(f"Fetching data for {indicator_name}...")
        raw_data = fetch_who_measure_data(indicator_code, country_groups, countries)
        
        if not raw_data:
            return "Error: No data received from WHO API"
        
        # Parse to DataFrame
        print("Parsing data to DataFrame...")
        df = parse_who_json_to_dataframe(raw_data)
        
        if df.empty:
            return "Error: No data available for the specified parameters"
        
        # Create the plot
        print("Creating plot...")
        plt.figure(figsize=(12, 7))
        
        # Define colors for different lines
        colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#7209B7', '#048A81', '#F77F00', '#8B5A2B', '#2D5016', '#6A0572']
        
        plot_entities = []
        entity_labels = []
        
        # Handle countries
        if countries and 'COUNTRY' in df.columns:
            country_data = df[df['COUNTRY'].isin(countries)]
            for country in countries:
                if country in country_data['COUNTRY'].values:
                    plot_entities.append(('COUNTRY', country))
                    # Get full country name from the data or use ISO3 as fallback
                    country_name = get_country_full_name(country)
                    entity_labels.append(country_name)
        
        # Handle country groups
        if country_groups and 'COUNTRY_GRP' in df.columns:
            group_data = df[df['COUNTRY_GRP'].isin(country_groups)]
            for group in country_groups:
                if group in group_data['COUNTRY_GRP'].values:
                    plot_entities.append(('COUNTRY_GRP', group))
                    entity_labels.append(f"{group} (Group)")
        
        # If no specific entities requested, plot all available
        if not plot_entities:
            if 'COUNTRY' in df.columns:
                unique_countries = df['COUNTRY'].unique()
                for country in unique_countries:
                    if country:  # Skip empty
                        plot_entities.append(('COUNTRY', country))
                        country_name = get_country_full_name(country)
                        entity_labels.append(country_name)
        
        # Plot lines for each entity
        for i, (entity_type, entity_value) in enumerate(plot_entities):
            if entity_type == 'COUNTRY':
                entity_data = df[df['COUNTRY'] == entity_value].copy()
            else:  # COUNTRY_GRP
                entity_data = df[df['COUNTRY_GRP'] == entity_value].copy()
            
            entity_data = entity_data.sort_values('YEAR')
            
            if not entity_data.empty and 'VALUE_NUMERIC' in entity_data.columns:
                color = colors[i % len(colors)]
                label = entity_labels[i] if i < len(entity_labels) else entity_value
                
                plt.plot(entity_data['YEAR'], entity_data['VALUE_NUMERIC'], 
                        marker='o', linewidth=2, markersize=4, 
                        color=color, label=label)
                
                # Add value labels on data points (every few points to avoid clutter)
                for idx, row in entity_data.iterrows():
                    if idx % 3 == 0:  # Show every 3rd point
                        plt.annotate(f'{row["VALUE_NUMERIC"]:.1f}', 
                                   (row['YEAR'], row['VALUE_NUMERIC']), 
                                   textcoords="offset points", 
                                   xytext=(0,8), ha='center', fontsize=8)
        
        # Customize the plot
        title_parts = []
        if countries:
            title_parts.append(f"Countries: {', '.join(countries)}")
        if country_groups:
            title_parts.append(f"Groups: {', '.join(country_groups)}")
        
        if title_parts:
            title = f'{indicator_name}\n({", ".join(title_parts)})'
        else:
            title = f'{indicator_name} by Country/Group'
            
        plt.title(title, fontsize=14, fontweight='bold', pad=15)
        plt.xlabel('Year', fontsize=11)
        plt.ylabel(indicator_name, fontsize=11)
        plt.grid(True, alpha=0.3)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # Set x-axis to show years properly
        if 'YEAR' in df.columns:
            years = sorted(df['YEAR'].dropna().unique())
            if len(years) > 10:
                # Show every nth year if too many
                step = max(1, len(years) // 10)
                plt.xticks(years[::step])
            else:
                plt.xticks(years)
        
        plt.tight_layout()
        
        # Save to BytesIO buffer
        buffer = BytesIO()
        plt.savefig(buffer, format='png', dpi=300, bbox_inches='tight')
        buffer.seek(0)
        image_data = buffer.getvalue()
        plt.close()
        
        # Generate filename with timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        safe_indicator = indicator_code.replace('/', '_').replace(' ', '_')
        filename = f"{safe_indicator}_{timestamp}.png"
        
        # Upload to Dropbox and get direct URL
        print("Uploading to Dropbox...")
        direct_url = upload_and_display_image(image_data, filename)
        
        if direct_url.startswith("Error:"):
            return direct_url
        
        print(f"Plot created and uploaded successfully!")
        return direct_url
        
    except Exception as e:
        return f"Error creating plot: {str(e)}"


In [10]:
def get_direct_image_url_from_dropbox(file_path):
    """Get direct image URL from Dropbox using temporary link"""
    try:
        # Try OAuth with refresh token first (recommended)
        app_key = os.getenv('DROPBOX_APP_KEY')
        app_secret = os.getenv('DROPBOX_APP_SECRET')
        refresh_token = os.getenv('DROPBOX_REFRESH_TOKEN')
        
        if app_key and app_secret and refresh_token:
            # Use OAuth with refresh token (automatically refreshes when expired)
            dbx = dropbox.Dropbox(
                app_key=app_key,
                app_secret=app_secret,
                oauth2_refresh_token=refresh_token
            )
        else:
            # Fallback to simple access token (will expire after 4 hours)
            access_token = os.getenv('DROPBOX_ACCESS_TOKEN')
            if not access_token:
                return "Error: DROPBOX_ACCESS_TOKEN or OAuth credentials not found"
            dbx = dropbox.Dropbox(access_token)
        # Get temporary direct link (valid for 4 hours)
        result = dbx.files_get_temporary_link(file_path)
        if result and hasattr(result, 'link'):
            return result.link
        else:
            return "Error: Could not get temporary link"
        
    except Exception as e:
        return f"Error getting direct URL: {str(e)}"

In [11]:


def upload_and_display_image(image_data, filename):
    """Complete workflow: upload to Dropbox and return direct image URL"""
    # Upload to Dropbox
    dropbox_url = upload_image_to_dropbox(image_data, filename)
    
    if dropbox_url.startswith("Error:"):
        return dropbox_url
    
    # Get direct temporary link for better chat display
    file_path = f"/{filename}"
    direct_url = get_direct_image_url_from_dropbox(file_path)
    
    if direct_url.startswith("Error:"):
        # Fallback to URL conversion method
        direct_url = convert_dropbox_url_to_direct(dropbox_url)
    
    # Return direct URL for chat display
    return direct_url


In [12]:
# Update tools list to include the new create_plot function
gw_tools = [search_indicators_advanced, create_plot,search_ind_name]

In [13]:
with open("intro/euro_countries.json", "r", encoding="utf-8") as f:
    euro_countries = f.read()

with open("intro/euro_cntry_grp.json", "r", encoding="utf-8") as f:
    euro_cntry_groups = f.read()
    
with open("intro/cntry_grp_map.csv", "r", encoding="utf-8") as f:
    cntry_grp_map = f.read()

In [14]:
with open("intro/about.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [15]:
agent_name = "European Health Information Gateway"
search_indicator = "search_indicators_advanced"
create_image = "create_plot"

In [16]:
# create model client autogen
from autogen_ext.models.openai import OpenAIChatCompletionClient
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

In [ ]:
system_prompt = f"You are acting as {agent_name}. You are answering questions on {agent_name}'s data portal, \
particularly questions related to {agent_name}'s health data, background, provide the links to health indicators from tool {search_indicator} \
Your responsibility is to represent {agent_name} for interactions on the website as faithfully as possible. \
    You task is to  guides users around gateway.euro.who.int and also helps them analyze and compare data \
You are given a table of {agent_name}'s indicators links in tool {search_indicator} summary about the portal which you can use to answer questions.\
    To find the indicator code for the indicator name, use tool {search_ind_name}.\
Use the tool {create_image} to create the image the return URL to display the png image in the chat. IMPORTANT provide the link to download the image!The link should be visible bellow the image.\
Be professional and engaging, as if talking to data health scientist or a usual person who came across the website and interested in health data. \
If you don't know the answer, say so. \
IMPORTANT: ONLY provide URLs that are retrieved from the database using the search {search_indicator}. Do NOT provide any URLs from your training \
data as they may be outdated. Always use the {search_indicator} tool to get current URLS and display up to five links"

system_prompt += f"\n\n## Summary:\n{summary}\n\n## health indicators list from tool {search_indicator}:please respond with links only from provided list\n\n \
## list of countries in WHO Europe in JSON:\n{euro_countries}\n\n \
## list of countries groups WHO Europe in JSON:\n{euro_cntry_groups}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {agent_name}."

In [18]:
system_prompt

'You are acting as European Health Information Gateway. You are answering questions on European Health Information Gateway\'s data portal, particularly questions related to European Health Information Gateway\'s health data, background, provide the links to health indicators from tool search_indicators_advanced Your responsibility is to represent European Health Information Gateway for interactions on the website as faithfully as possible.     You task is to  guides users around gateway.euro.who.int and also helps them analyze and compare data You are given a table of European Health Information Gateway\'s indicators links in tool search_indicators_advanced summary about the portal which you can use to answer questions.    To find the indicator code for the indicator name, use tool <function search_ind_name at 0x72e1163eb560>.Use the tool create_plot to create the image the return URL to display the png image in the chat and provide a link to download the imageBe professional and engag

In [23]:
from autogen_agentchat.agents import AssistantAgent

smart_agent = AssistantAgent(
    name='gw_chat',
    model_client=model_client,
    system_message= system_prompt,
    model_client_stream=True,
    tools=gw_tools,
    reflect_on_tool_use=True
)

In [24]:
import asyncio
from autogen_core import CancellationToken
from autogen_agentchat.messages import TextMessage

def chat(message, history):
    """Chat function that properly handles the smart agent in Jupyter environment"""
    try:
        # Create a new event loop in a separate thread to avoid conflicts with Jupyter's loop
        import concurrent.futures
        import threading
        
        def run_async():
            # Create a new event loop for this thread
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            try:
                return loop.run_until_complete(chat_async(message, history))
            finally:
                loop.close()
        
        # Run the async function in a separate thread
        with concurrent.futures.ThreadPoolExecutor() as executor:
            future = executor.submit(run_async)
            return future.result()
            
    except Exception as e:
        return f"Error: {str(e)}"

async def chat_async(message, history):
    """Async chat function that properly handles the smart agent"""
    try:
        # Convert string message to TextMessage object that autogen expects
        text_message = TextMessage(content=message, source="user")
        response = await smart_agent.on_messages([text_message], cancellation_token=CancellationToken())
        # Access the content from the response
        return response.chat_message.content
    except Exception as e:
        return f"Error: {str(e)}"

In [25]:
# Launch the Gradio chat interface with proper message format
interface = gr.ChatInterface(
    fn=chat,
    title="European Health Information Gateway Chat",
    description="Ask questions about health indicators and data from the WHO European Region",
    examples=[
        "Plot Life Expectancy at Birth for Denmark and the WHO European Region",
        "Show me data about life expectancy",
        "Show maternal mortality rates in  WHO European Region",
        "Compare the estimated maternal mortality ratios for both the EU and the WHO European Region"
    ],
    type="messages"  # Use the new message format to avoid deprecation warning
)

interface.launch(share=False)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Fetching data for Life Expectancy at Birth...
Establishing session with https://dw.euro.who.int...
Making API request to https://dw.euro.who.int/api/v3/measures/hfa_43?filter=COUNTRY_GRP:WHO_EURO;COUNTRY:DNK...
Parsing data to DataFrame...
Creating plot...
Uploading to Dropbox...
Plot created and uploaded successfully!


Task was destroyed but it is pending!
task: <Task pending name='Task-1233' coro=<<async_generator_athrow without __name__>()>>
Task was destroyed but it is pending!
task: <Task pending name='Task-1630' coro=<<async_generator_athrow without __name__>()>>
